In [ ]:
import pandas as pd
from io import StringIO
import boto3
import os

s3 = boto3.client("s3")

VALID_STATUS = ["SUCCESS", "FAILED", "PENDING"]


def clean_transactions(df):

    # ----------------------
    # Remove duplicates
    # ----------------------
    df = df.drop_duplicates(
        subset=["transaction_id"],
        keep="first"
    )

    # ----------------------
    # Standardize string columns
    # ----------------------
    string_cols = [
        "merchant",
        "currency",
        "payment_method",
        "status"
    ]

    for col in string_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

    # ----------------------
    # Convert timestamps
    # ----------------------
    df["transaction_time"] = pd.to_datetime(
        df["transaction_time"],
        errors="coerce",
        dayfirst=True
    )
    # Merchant
    df["merchant"] = df["merchant"].fillna("UNKNOWN_MERCHANT")
    df["merchant"] = df["merchant"].replace(
    ["NONE", "NAN", ""],
    "UNKNOWN_MERCHANT"
    )
    # Status
    df["status"] = df["status"].fillna("UNKNOWN_STATUS")
    df["status"] = df["status"].replace(
    ["NONE", "NAN", ""],
    "UNKNOWN_STATUS"
    )
    # Currency
    df["currency"] = df["currency"].fillna("UNKNOWN_CURRENCY")
    df["currency"] = df["currency"].replace(
    ["NONE", "NAN", ""],
    "UNKNOWN_CURRENCY"
    )
    # Payment Method
    df["payment_method"] = df["payment_method"].fillna(
    "UNKNOWN_PAYMENT_METHOD"
    )
    df["payment_method"] = df["payment_method"].replace(
    ["NONE", "NAN", ""],
    "UNKNOWN_PAYMENT_METHOD"
    )

    # ----------------------
    # Validation flags
    # ----------------------

    # Amount is critical
    invalid_amount = (
        df["amount"].isna()
        | (df["amount"] <= 0)
    )

    # Timestamp is critical
    invalid_timestamp = (
        df["transaction_time"].isna()
    )

    # Status validation
    invalid_status = (
        ~df["status"].isin(
            VALID_STATUS + ["UNKNOWN_STATUS"]
        )
    )

    # ----------------------
    # Quarantine records
    # ----------------------
    quarantine_mask = (
        invalid_amount
        | invalid_timestamp
        | invalid_status
    )

    quarantine_df = df[quarantine_mask].copy()

    quarantine_df["reason"] = ""

    quarantine_df.loc[
        invalid_amount,
        "reason"
    ] += "INVALID_AMOUNT;"

    quarantine_df.loc[
        invalid_timestamp,
        "reason"
    ] += "INVALID_TIMESTAMP;"

    quarantine_df.loc[
        invalid_status,
        "reason"
    ] += "INVALID_STATUS;"

    # ----------------------
    # Clean records
    # ----------------------
    clean_df = df[~quarantine_mask].copy()

    return clean_df, quarantine_df


def lambda_handler(event, context):

    # Get bucket and key from S3 trigger
    bucket = event["Records"][0]["s3"]["bucket"]["name"]
    key = event["Records"][0]["s3"]["object"]["key"]

    print(f"Bucket: {bucket}")
    print(f"Key: {key}")

    # Read CSV from S3
    obj = s3.get_object(
        Bucket=bucket,
        Key=key
    )

    df = pd.read_csv(obj["Body"])

    # Clean data
    clean_df, quarantine_df = clean_transactions(df)

    filename = os.path.basename(key)

    cleaned_key = f"cleaned/{filename}"
    quarantine_key = f"quarantine/{filename}"

    print(f"Cleaned Key: {cleaned_key}")
    print(f"Quarantine Key: {quarantine_key}")

    # ----------------------
    # Save cleaned data
    # ----------------------
    clean_buffer = StringIO()

    clean_df.to_csv(
        clean_buffer,
        index=False
    )

    s3.put_object(
        Bucket=bucket,
        Key=cleaned_key,
        Body=clean_buffer.getvalue()
    )

    # ----------------------
    # Save quarantine data
    # ----------------------
    quarantine_buffer = StringIO()

    quarantine_df.to_csv(
        quarantine_buffer,
        index=False
    )

    s3.put_object(
        Bucket=bucket,
        Key=quarantine_key,
        Body=quarantine_buffer.getvalue()
    )

    print(f"Total Records: {len(df)}")
    print(f"Clean Records: {len(clean_df)}")
    print(f"Quarantine Records: {len(quarantine_df)}")

    return {
        "statusCode": 200,
        "total_records": len(df),
        "clean_records": len(clean_df),
        "quarantined_records": len(quarantine_df)
    }